# HW8 Experiment: Scaled Non-IID + Client Failures (30 Clients)

**Project:** Nexus — LLM-Orchestrated Federated Learning  
**Course:** MAS.664 AI Studio, MIT Media Lab, Spring 2026  

## Motivation (building on HW7)
HW7 found that the agent showed **no advantage over random selection with IID data**. The hypothesis was: *"The agent's value would emerge under more realistic conditions."*

This experiment tests that hypothesis directly:
- **Scale:** 10 clients → 30 clients (3× scale-up)
- **Non-IID data:** Each client holds only 2–3 CIFAR-10 classes (vs. uniform IID in HW7)
- **Client failures:** 15% of clients randomly fail each round (return no update)
- **Question:** Does the agent finally outperform random selection when data quality varies AND failures occur?

**Cloud note:** This notebook runs on Google Colab (Google Cloud infrastructure). Flower's simulation engine spawns each of the 30 clients as an isolated virtual process — equivalent to 30 agents running on the cloud.

## Setup

In [ ]:
# Install dependencies
!pip install flwr[simulation] torch torchvision anthropic --quiet

In [ ]:
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
import json
import time
import os
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Dict, Optional, Tuple

print(f"Flower version: {fl.__version__}")
print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Configuration

In [ ]:
# ── Experiment config ──────────────────────────────────────────────
NUM_CLIENTS        = 30        # Scale-up from HW7's 10
NUM_ROUNDS         = 15        # Same as HW7 for comparability
CLIENTS_PER_ROUND  = 10        # Select 10 of 30 each round
FAILURE_RATE       = 0.15      # 15% of selected clients fail each round
CLASSES_PER_CLIENT = 2         # Non-IID: each client gets 2 CIFAR-10 classes
LOCAL_EPOCHS       = 1
BATCH_SIZE         = 32
RANDOM_SEED        = 42

# ── Anthropic API key (optional — falls back to heuristic) ─────────
ANTHROPIC_API_KEY  = ""        # Paste your key here, or leave blank for heuristic mode

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

CIFAR10_CLASSES = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]
print("Config loaded.")
print(f"  {NUM_CLIENTS} clients | {NUM_ROUNDS} rounds | {CLIENTS_PER_ROUND} selected/round")
print(f"  Failure rate: {FAILURE_RATE*100:.0f}% | Classes/client: {CLASSES_PER_CLIENT}")

## Non-IID Data Partitioning

Each client receives samples from only `CLASSES_PER_CLIENT` CIFAR-10 classes. This creates realistic data heterogeneity — some clients have complementary data (unique classes), others have overlapping or redundant data.

In [ ]:
def create_noniid_partitions(dataset, num_clients: int, classes_per_client: int, seed: int = 42):
    """
    Assign each client a random subset of classes.
    Returns list of index arrays, one per client.
    Also returns which classes each client holds (for agent context).
    """
    rng = np.random.default_rng(seed)
    num_classes = 10  # CIFAR-10

    # Group dataset indices by class
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset):
        class_indices[label].append(idx)

    client_indices = []
    client_classes = []  # Which classes each client holds

    for c in range(num_clients):
        # Assign classes_per_client random classes to this client
        assigned = rng.choice(num_classes, size=classes_per_client, replace=False).tolist()
        client_classes.append(assigned)

        # Collect indices for those classes
        indices = []
        for cls in assigned:
            cls_idx = class_indices[cls]
            # Each client gets a chunk of each class
            per_client_per_class = len(cls_idx) // num_clients
            start = c * per_client_per_class
            end   = start + per_client_per_class
            indices.extend(cls_idx[start:end])

        rng.shuffle(indices)
        client_indices.append(indices)

    return client_indices, client_classes


# Download CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=128, shuffle=False)

# Partition
client_indices, client_classes = create_noniid_partitions(
    trainset, NUM_CLIENTS, CLASSES_PER_CLIENT, RANDOM_SEED
)

print(f"Non-IID partitioning complete.")
print(f"  Sample sizes: min={min(len(i) for i in client_indices)}, "
      f"max={max(len(i) for i in client_indices)}, "
      f"mean={np.mean([len(i) for i in client_indices]):.0f}")
print()
print("Client class assignments (first 10 clients):")
for c in range(10):
    class_names = [CIFAR10_CLASSES[cls] for cls in client_classes[c]]
    print(f"  Client {c:02d}: {class_names}")

## Model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 10)
        self.relu  = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)


def get_parameters(model):
    return [val.cpu().numpy() for val in model.state_dict().values()]

def set_parameters(model, parameters):
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict  = {k: torch.tensor(v) for k, v in params_dict}
    model.load_state_dict(state_dict, strict=True)


def train(model, dataloader, epochs=1):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    for _ in range(epochs):
        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    return total_loss / len(dataloader)


def evaluate(model, dataloader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, total_loss = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total, total_loss / len(dataloader)


print("Model defined.")

## Agent (Claude API + Heuristic Fallback)

The agent receives per-client metadata each round (which classes each client holds, past accuracy contributions) and decides which 10 clients to select. It also flags clients that have failed repeatedly.

In [ ]:
class NexusAgent:
    """
    LLM-powered orchestrator for federated learning.
    Falls back to heuristic if no API key is set.
    """

    def __init__(self, api_key: str, num_clients: int, clients_per_round: int,
                 client_classes: List[List[int]]):
        self.api_key          = api_key
        self.num_clients      = num_clients
        self.clients_per_round = clients_per_round
        self.client_classes   = client_classes  # classes each client holds
        self.history          = []              # round logs
        self.failure_counts   = defaultdict(int)
        self.api_latencies    = []  # track API cost

    def decide(self, round_num: int, available_clients: List[int],
               prev_accuracies: Dict[int, float]) -> Dict:
        """
        Returns: {selected_clients, reasoning, strategy, api_latency_ms}
        """
        if self.api_key:
            return self._llm_decide(round_num, available_clients, prev_accuracies)
        else:
            return self._heuristic_decide(round_num, available_clients, prev_accuracies)

    def _build_prompt(self, round_num, available_clients, prev_accuracies):
        client_info = []
        for cid in available_clients:
            classes = [CIFAR10_CLASSES[c] for c in self.client_classes[cid]]
            acc = prev_accuracies.get(cid, None)
            fails = self.failure_counts[cid]
            client_info.append(
                f"Client {cid}: classes={classes}, past_contribution={acc:.3f if acc else 'N/A'}, failures={fails}"
            )

        history_str = ""
        if self.history:
            last = self.history[-1]
            history_str = f"Last round accuracy: {last['accuracy']:.3f}"

        return f"""You are orchestrating a federated learning system. It is round {round_num} of {NUM_ROUNDS}.

Available clients this round:
{chr(10).join(client_info)}

{history_str}

Goal: Select {self.clients_per_round} clients that maximize class diversity and avoid unreliable clients.
Non-IID context: clients with more unique/rare classes are more valuable.
Failures: clients with high failure counts should be deprioritized.

Respond ONLY with valid JSON:
{{
  "selected_clients": [list of {self.clients_per_round} client IDs],
  "reasoning": "brief explanation",
  "strategy": "diversity|reliability|balanced"
}}"""

    def _llm_decide(self, round_num, available_clients, prev_accuracies):
        import anthropic
        client = anthropic.Anthropic(api_key=self.api_key)
        prompt = self._build_prompt(round_num, available_clients, prev_accuracies)

        start = time.time()
        try:
            msg = client.messages.create(
                model="claude-opus-4-5",
                max_tokens=512,
                messages=[{"role": "user", "content": prompt}]
            )
            latency_ms = (time.time() - start) * 1000
            self.api_latencies.append(latency_ms)

            raw = msg.content[0].text.strip()
            # Strip markdown fences if present
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            result = json.loads(raw)

            # Validate and clip to available clients
            selected = [c for c in result["selected_clients"] if c in available_clients]
            # Fill if too few
            if len(selected) < self.clients_per_round:
                remaining = [c for c in available_clients if c not in selected]
                selected += random.sample(remaining, self.clients_per_round - len(selected))
            selected = selected[:self.clients_per_round]

            return {
                "selected_clients": selected,
                "reasoning": result.get("reasoning", ""),
                "strategy": result.get("strategy", "unknown"),
                "api_latency_ms": latency_ms,
                "mode": "llm"
            }
        except Exception as e:
            print(f"  [Agent] LLM error: {e} — falling back to heuristic")
            return self._heuristic_decide(round_num, available_clients, prev_accuracies)

    def _heuristic_decide(self, round_num, available_clients, prev_accuracies):
        """
        Heuristic: prefer clients with diverse classes and low failure counts.
        Score = class_diversity_bonus - failure_penalty
        """
        # Count class frequency across all available clients to find rare classes
        class_freq = defaultdict(int)
        for cid in available_clients:
            for cls in self.client_classes[cid]:
                class_freq[cls] += 1

        def score(cid):
            # Rarer classes = higher score
            diversity = sum(1.0 / class_freq[cls] for cls in self.client_classes[cid])
            reliability = -self.failure_counts[cid] * 0.5
            return diversity + reliability

        scored = sorted(available_clients, key=score, reverse=True)
        selected = scored[:self.clients_per_round]

        return {
            "selected_clients": selected,
            "reasoning": "Heuristic: diversity-weighted, failure-penalized selection",
            "strategy": "diversity",
            "api_latency_ms": 0,
            "mode": "heuristic"
        }

    def log_round(self, round_num, selected, decision, failed_this_round,
                  accuracy, loss, round_time_s):
        for cid in failed_this_round:
            self.failure_counts[cid] += 1
        entry = {
            "round": round_num,
            "selected": selected,
            "failed": failed_this_round,
            "actually_trained": [c for c in selected if c not in failed_this_round],
            "accuracy": accuracy,
            "loss": loss,
            "round_time_s": round_time_s,
            "reasoning": decision["reasoning"],
            "strategy": decision["strategy"],
            "api_latency_ms": decision["api_latency_ms"],
            "mode": decision["mode"]
        }
        self.history.append(entry)
        return entry


print("NexusAgent defined.")

## FL Simulation Engine

A lightweight custom FL loop (no Flower server/client boilerplate needed for simulation — we control the loop directly). Each round:
1. Agent selects clients
2. Some clients "fail" (simulated dropout)
3. Surviving clients train locally
4. Global model aggregated via FedAvg
5. Evaluate on test set

In [ ]:
def fedavg_aggregate(global_params, client_params_list, client_weights):
    """
    Weighted average of client parameters.
    client_weights: number of samples each client trained on.
    """
    total_weight = sum(client_weights)
    aggregated = []
    for layer_idx in range(len(global_params)):
        layer_avg = sum(
            (w / total_weight) * params[layer_idx]
            for params, w in zip(client_params_list, client_weights)
        )
        aggregated.append(layer_avg)
    return aggregated


def run_fl_experiment(strategy: str, seed: int = 42) -> List[Dict]:
    """
    strategy: 'agent' or 'random'
    Returns list of per-round logs.
    """
    print(f"\n{'='*60}")
    print(f"Running: {strategy.upper()} strategy")
    print(f"{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Global model
    global_model = SimpleCNN().to(DEVICE)
    global_params = get_parameters(global_model)

    # Per-client accuracy tracking (for agent context)
    client_accuracy_history = defaultdict(list)
    prev_accuracies = {}

    # Agent (only used if strategy=='agent')
    agent = NexusAgent(
        api_key=ANTHROPIC_API_KEY,
        num_clients=NUM_CLIENTS,
        clients_per_round=CLIENTS_PER_ROUND,
        client_classes=client_classes
    )

    logs = []

    for rnd in range(1, NUM_ROUNDS + 1):
        round_start = time.time()
        all_clients = list(range(NUM_CLIENTS))

        # ── Client selection ─────────────────────────────────────────
        if strategy == 'agent':
            decision = agent.decide(rnd, all_clients, prev_accuracies)
            selected = decision['selected_clients']
        else:
            selected = random.sample(all_clients, CLIENTS_PER_ROUND)
            decision = {
                "selected_clients": selected, "reasoning": "Random selection",
                "strategy": "random", "api_latency_ms": 0, "mode": "random"
            }

        # ── Simulate failures ─────────────────────────────────────────
        failed = []
        survived = []
        for cid in selected:
            if random.random() < FAILURE_RATE:
                failed.append(cid)
            else:
                survived.append(cid)

        if not survived:
            print(f"  Round {rnd:02d}: ALL clients failed — skipping round")
            logs.append({"round": rnd, "accuracy": logs[-1]["accuracy"] if logs else 0.1,
                         "loss": 9.9, "num_failed": len(failed), "num_trained": 0,
                         "selected": selected, "failed": failed})
            continue

        # ── Local training ────────────────────────────────────────────
        client_params_list = []
        client_weights     = []
        client_round_acc   = {}

        for cid in survived:
            # Load client data
            subset = Subset(trainset, client_indices[cid])
            loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

            # Load global params into local model
            local_model = SimpleCNN().to(DEVICE)
            set_parameters(local_model, global_params)

            # Train
            train_loss = train(local_model, loader, LOCAL_EPOCHS)
            local_acc, _ = evaluate(local_model, testloader)

            client_params_list.append(get_parameters(local_model))
            client_weights.append(len(subset))
            client_round_acc[cid] = local_acc

        # ── Aggregation ───────────────────────────────────────────────
        global_params = fedavg_aggregate(global_params, client_params_list, client_weights)
        set_parameters(global_model, global_params)

        # ── Evaluation ────────────────────────────────────────────────
        global_acc, global_loss = evaluate(global_model, testloader)

        # Update prev_accuracies for agent next round
        prev_accuracies.update(client_round_acc)

        round_time = time.time() - round_start

        # ── Logging ───────────────────────────────────────────────────
        if strategy == 'agent':
            log = agent.log_round(rnd, selected, decision, failed,
                                  global_acc, global_loss, round_time)
        else:
            log = {
                "round": rnd, "selected": selected, "failed": failed,
                "actually_trained": survived, "accuracy": global_acc,
                "loss": global_loss, "round_time_s": round_time,
                "reasoning": "Random", "strategy": "random",
                "api_latency_ms": 0, "mode": "random",
                "num_failed": len(failed), "num_trained": len(survived)
            }
        log["num_failed"] = len(failed)
        log["num_trained"] = len(survived)
        logs.append(log)

        print(f"  Round {rnd:02d} | acc={global_acc:.4f} loss={global_loss:.4f} "
              f"| trained={len(survived)}/{len(selected)} "
              f"| failed={len(failed)} "
              f"| {round_time:.1f}s")

    return logs, agent if strategy == 'agent' else None


print("FL simulation engine defined.")

## Run Experiments

Both conditions run with identical seeds, data, and failure events for fair comparison.

In [ ]:
# Run RANDOM baseline
random_logs, _ = run_fl_experiment('random', seed=RANDOM_SEED)

In [ ]:
# Run AGENT-guided
agent_logs, agent_obj = run_fl_experiment('agent', seed=RANDOM_SEED)

## Results & Analysis

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────
def summarize(logs, label):
    accs  = [l['accuracy'] for l in logs]
    fails = [l['num_failed'] for l in logs]
    times = [l['round_time_s'] for l in logs]
    print(f"\n{label}")
    print(f"  Final accuracy:    {accs[-1]:.4f}")
    print(f"  Best accuracy:     {max(accs):.4f}")
    print(f"  Avg failures/round:{np.mean(fails):.2f}")
    print(f"  Total failures:    {sum(fails)}")
    print(f"  Avg round time:    {np.mean(times):.2f}s")
    return accs, fails, times

random_accs, random_fails, random_times = summarize(random_logs, "RANDOM")
agent_accs,  agent_fails,  agent_times  = summarize(agent_logs,  "AGENT")

if agent_obj and agent_obj.api_latencies:
    print(f"\nAgent API latency:")
    print(f"  Mean: {np.mean(agent_obj.api_latencies):.0f}ms")
    print(f"  Max:  {max(agent_obj.api_latencies):.0f}ms")
    print(f"  Total API calls: {len(agent_obj.api_latencies)}")

In [ ]:
# ── Visualizations ─────────────────────────────────────────────────
rounds = list(range(1, NUM_ROUNDS + 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f'HW8: Nexus Agent vs Random — 30 Clients, Non-IID, {FAILURE_RATE*100:.0f}% Failure Rate',
    fontsize=13, fontweight='bold'
)

# (1) Accuracy over rounds
ax = axes[0, 0]
ax.plot(rounds, random_accs, 'b-o', label='Random', linewidth=2, markersize=5)
ax.plot(rounds, agent_accs,  'r-s', label='Agent',  linewidth=2, markersize=5)
ax.set_title('Global Accuracy per Round')
ax.set_xlabel('Round')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# (2) Failures per round
ax = axes[0, 1]
ax.bar([r - 0.2 for r in rounds], random_fails, 0.35, label='Random', color='blue', alpha=0.6)
ax.bar([r + 0.2 for r in rounds], agent_fails,  0.35, label='Agent',  color='red',  alpha=0.6)
ax.axhline(CLIENTS_PER_ROUND * FAILURE_RATE, color='gray', linestyle='--',
           label=f'Expected ({FAILURE_RATE*100:.0f}%)')
ax.set_title('Client Failures per Round')
ax.set_xlabel('Round')
ax.set_ylabel('# Failed Clients')
ax.legend()
ax.grid(True, alpha=0.3)

# (3) Round times
ax = axes[1, 0]
ax.plot(rounds, random_times, 'b-o', label='Random', linewidth=2, markersize=5)
ax.plot(rounds, agent_times,  'r-s', label='Agent',  linewidth=2, markersize=5)
if agent_obj and agent_obj.api_latencies:
    api_times = [l / 1000 for l in agent_obj.api_latencies]  # ms → s
    ax.plot(rounds[:len(api_times)], api_times, 'r--', label='API latency only', alpha=0.5)
ax.set_title('Round Wall-Clock Time (s)')
ax.set_xlabel('Round')
ax.set_ylabel('Seconds')
ax.legend()
ax.grid(True, alpha=0.3)

# (4) Accuracy gap (agent - random)
ax = axes[1, 1]
gap = [a - r for a, r in zip(agent_accs, random_accs)]
colors = ['green' if g > 0 else 'red' for g in gap]
ax.bar(rounds, gap, color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Agent Accuracy Advantage (Agent − Random)')
ax.set_xlabel('Round')
ax.set_ylabel('Accuracy Difference')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hw8_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: hw8_results.png")

In [ ]:
# ── Class diversity analysis ───────────────────────────────────────
# For agent runs: which classes did the agent prefer to select?
print("Class coverage analysis (agent selection):")
class_selection_count = defaultdict(int)
for log in agent_logs:
    for cid in log.get('actually_trained', []):
        for cls in client_classes[cid]:
            class_selection_count[cls] += 1

print("  Class name       | Times selected")
print("  " + "-" * 35)
for cls_id in range(10):
    print(f"  {CIFAR10_CLASSES[cls_id]:16s} | {class_selection_count[cls_id]}")

print("\nClass coverage analysis (random selection):")
class_selection_count_r = defaultdict(int)
for log in random_logs:
    for cid in log.get('actually_trained', []):
        for cls in client_classes[cid]:
            class_selection_count_r[cls] += 1

print("  Class name       | Times selected")
print("  " + "-" * 35)
for cls_id in range(10):
    print(f"  {CIFAR10_CLASSES[cls_id]:16s} | {class_selection_count_r[cls_id]}")

In [ ]:
# ── Agent reasoning log (last 5 rounds) ───────────────────────────
print("Agent reasoning — last 5 rounds:")
print("-" * 60)
for log in agent_logs[-5:]:
    print(f"Round {log['round']:02d} | strategy={log['strategy']} | acc={log['accuracy']:.4f}")
    print(f"  Reasoning: {log['reasoning'][:120]}")
    print(f"  Failed: {log['failed']} | API latency: {log['api_latency_ms']:.0f}ms")
    print()

In [ ]:
# ── Save full logs ─────────────────────────────────────────────────
output = {
    "config": {
        "num_clients": NUM_CLIENTS,
        "num_rounds": NUM_ROUNDS,
        "clients_per_round": CLIENTS_PER_ROUND,
        "failure_rate": FAILURE_RATE,
        "classes_per_client": CLASSES_PER_CLIENT
    },
    "random": random_logs,
    "agent": agent_logs
}

with open('hw8_logs.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)

print("Logs saved: hw8_logs.json")
print("\n" + "="*60)
print("EXPERIMENT COMPLETE")
print("="*60)
print(f"Final accuracy — Random: {random_accs[-1]:.4f} | Agent: {agent_accs[-1]:.4f}")
print(f"Accuracy delta: {agent_accs[-1] - random_accs[-1]:+.4f}")
total_random_fails = sum(random_fails)
total_agent_fails  = sum(agent_fails)
print(f"Total failures — Random: {total_random_fails} | Agent: {total_agent_fails}")

## Key Questions This Experiment Answers

1. **Does the agent outperform random with non-IID data?** (HW7 null result revisited)
2. **How does accuracy degrade under 15% client failure rate?**
3. **Does the agent adapt its selection strategy after observing failures?**
4. **What is the API latency overhead at 30-client scale?** (Cost of intelligence)
5. **Which classes get systematically under-represented under random vs agent selection?**

### Comparison with HW7 baseline:
| Metric | HW7 (IID, 10 clients) | HW8 (non-IID, 30 clients, failures) |
|--------|----------------------|--------------------------------------|
| Agent accuracy | ~50% | TBD |
| Random accuracy | ~50% | TBD |
| Agent advantage | 0% | TBD |
| Client failures | 0 | ~1.5/round (expected) |
| Agent overhead | - | API latency per round |